In [ ]:
from pathlib import Path
import os
import sys

PROJECT_DIR = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "steel_defect_pipeline.ipynb").is_file() and (path / "Defect in NEU.v2i.yolov8").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Direktori proyek tidak ditemukan.")
os.chdir(PROJECT_DIR)

SOURCE_DIR = PROJECT_DIR / "Defect in NEU.v2i.yolov8"
DATA_DIR = PROJECT_DIR / "outputs" / "5_split" / "dataset_801010"
EDA_DIR = PROJECT_DIR / "outputs" / "1_eda"
BALANCING_DIR = PROJECT_DIR / "outputs" / "2_balancing_notebook"
BALANCED_TRAIN_DIR = BALANCING_DIR / "train_balanced"
EVALUATE_DIR = PROJECT_DIR / "outputs" / "4_evaluate"
TRAIN_DIR = PROJECT_DIR / "outputs" / "3_train"
MODEL = "yolov8n.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
PATIENCE = 20
WEIGHTS_PATH = TRAIN_DIR / f"unbalanced_{Path(MODEL).stem}" / "weights" / "best.pt"


In [ ]:
if not (DATA_DIR / "data.yaml").is_file():
    sys.argv = ["split"]
    import argparse
    import collections
    import shutil
    from pathlib import Path

    from sklearn.model_selection import train_test_split


    PROJECT_DIR = Path.cwd().resolve()
    DEFAULT_SRC = PROJECT_DIR / "Defect in NEU.v2i.yolov8"
    DEFAULT_DST = PROJECT_DIR / "outputs" / "5_split" / "dataset_801010"
    IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


    def image_primary_class(image_path):
        """Return the most frequent class in the image for approximate stratification."""
        label_path = image_path.parent.parent / "labels" / f"{image_path.stem}.txt"
        if not label_path.is_file():
            return "__background__"

        classes = []
        with label_path.open("r", encoding="utf-8") as label_file:
            for line_number, line in enumerate(label_file, start=1):
                fields = line.split()
                if not fields:
                    continue
                if len(fields) != 5:
                    raise ValueError(f"Label harus memiliki 5 kolom: {label_path}:{line_number}")
                try:
                    classes.append(int(fields[0]))
                    for value in fields[1:]:
                        float(value)
                except ValueError as exc:
                    raise ValueError(f"Format label tidak valid: {label_path}:{line_number}") from exc

        if not classes:
            return "__background__"
        return collections.Counter(classes).most_common(1)[0][0]


    def split_items(items, labels, test_size, description):
        """Stratify by the most frequent class, falling back when strata are too small."""
        try:
            return train_test_split(
                items, labels, test_size=test_size, stratify=labels, random_state=42
            )
        except ValueError as exc:
            print(f"Peringatan: stratifikasi {description} tidak dapat dilakukan ({exc}).")
            print(f"Membagi {description} secara acak sebagai fallback.")
            return train_test_split(
                items, labels, test_size=test_size, random_state=42
            )


    def main():
        parser = argparse.ArgumentParser(description=__doc__)
        parser.add_argument("--src", type=Path, default=DEFAULT_SRC,
                            help="Folder dataset sumber (default: Defect in NEU.v2i.yolov8)")
        parser.add_argument("--dst", type=Path, default=DEFAULT_DST,
                            help="Folder keluaran (default: outputs/5_split/dataset_801010)")
        args = parser.parse_args()

        source_dir = args.src.resolve()
        destination_dir = args.dst.resolve()
        if not source_dir.is_dir():
            raise FileNotFoundError(f"Folder sumber tidak ditemukan: {source_dir}")
        if source_dir == destination_dir or source_dir in destination_dir.parents:
            raise ValueError("Folder keluaran harus berada di luar folder dataset sumber.")
        if destination_dir.exists() and any(destination_dir.iterdir()):
            raise FileExistsError(
                f"Folder keluaran sudah ada dan tidak kosong: {destination_dir}. "
                "Pilih folder lain dengan --dst."
            )

        images = []
        for split in ("train", "valid", "test"):
            images_dir = source_dir / split / "images"
            if not images_dir.is_dir():
                raise FileNotFoundError(f"Folder gambar tidak ditemukan: {images_dir}")
            images.extend(
                path for path in images_dir.iterdir()
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            )

        images.sort()
        if len(images) < 3:
            raise ValueError(f"Minimal 3 gambar diperlukan untuk split; ditemukan {len(images)}.")
        image_names = [image.name for image in images]
        if len(image_names) != len(set(image_names)):
            raise ValueError("Nama file gambar duplikat ditemukan di beberapa split sumber.")

        labels = [image_primary_class(image) for image in images]
        train_images, remainder_images, train_labels, remainder_labels = split_items(
            images, labels, test_size=0.2, description="train dan sisa data"
        )
        valid_images, test_images, _, _ = split_items(
            remainder_images, remainder_labels, test_size=0.5,
            description="valid dan test"
        )

        for split_name, split_images in (
            ("train", train_images),
            ("valid", valid_images),
            ("test", test_images),
        ):
            images_dir = destination_dir / split_name / "images"
            labels_dir = destination_dir / split_name / "labels"
            images_dir.mkdir(parents=True, exist_ok=True)
            labels_dir.mkdir(parents=True, exist_ok=True)

            for image in split_images:
                shutil.copy2(image, images_dir / image.name)
                label = image.parent.parent / "labels" / f"{image.stem}.txt"
                if label.is_file():
                    shutil.copy2(label, labels_dir / label.name)

            print(f"{split_name}: {len(split_images)} gambar")

        source_yaml = source_dir / "data.yaml"
        if source_yaml.is_file():
            shutil.copy2(source_yaml, destination_dir / "data.yaml")

        print(f"\nDataset split tersimpan di: {destination_dir}")
        print("Rasio target: 80% train, 10% valid, 10% test.")
        print("Stratifikasi berdasarkan kelas yang paling sering muncul per gambar.")
    main()

for split_name in ("train", "valid", "test"):
    for subfolder in ("images", "labels"):
        folder = DATA_DIR / split_name / subfolder
        if not folder.is_dir():
            raise FileNotFoundError(f"Folder dataset tidak ditemukan: {folder}")


In [ ]:
import sys
sys.argv = ["eda", "--data_dir", str(DATA_DIR), "--out_dir", str(EDA_DIR)]

import os
import argparse
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

PROJECT_DIR = Path.cwd().resolve()
DEFAULT_DATA_DIR = PROJECT_DIR / "outputs" / "5_split" / "dataset_801010"
DEFAULT_OUT_DIR = PROJECT_DIR / "outputs" / "1_eda"


def load_class_names(data_yaml_path):
    with open(data_yaml_path, "r") as f:
        data = yaml.safe_load(f)
    return data["names"]


def parse_split(split_dir, class_names):
    """
    Membaca semua file label di satu split (train/valid/test),
    mengembalikan DataFrame dengan satu baris per bounding box.
    """
    images_dir = os.path.join(split_dir, "images")
    labels_dir = os.path.join(split_dir, "labels")

    rows = []
    image_files = [f for f in os.listdir(images_dir)
                   if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    for img_file in image_files:
        label_file = os.path.splitext(img_file)[0] + ".txt"
        label_path = os.path.join(labels_dir, label_file)

        # Ambil dimensi gambar asli (untuk hitung ukuran box dalam piksel)
        img_path = os.path.join(images_dir, img_file)
        try:
            with Image.open(img_path) as im:
                img_w, img_h = im.size
        except Exception:
            img_w, img_h = None, None

        if not os.path.exists(label_path):
            # Gambar tanpa anotasi (jarang di NEU-DET, tapi dijaga)
            rows.append({
                "image": img_file, "class_id": None, "class_name": None,
                "box_w_norm": None, "box_h_norm": None,
                "box_w_px": None, "box_h_px": None,
            })
            continue

        with open(label_path, "r") as f:
            lines = f.readlines()

        if len(lines) == 0:
            rows.append({
                "image": img_file, "class_id": None, "class_name": None,
                "box_w_norm": None, "box_h_norm": None,
                "box_w_px": None, "box_h_px": None,
            })

        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            class_id = int(parts[0])
            _, _, w_norm, h_norm = map(float, parts[1:5])

            row = {
                "image": img_file,
                "class_id": class_id,
                "class_name": class_names[class_id],
                "box_w_norm": w_norm,
                "box_h_norm": h_norm,
            }
            if img_w and img_h:
                row["box_w_px"] = w_norm * img_w
                row["box_h_px"] = h_norm * img_h
            rows.append(row)

    df = pd.DataFrame(rows)
    return df


def categorize_box_size(area_px, small_thresh=32**2, medium_thresh=96**2):
    """Kategori ukuran box ala COCO: small/medium/large (dalam piksel^2)."""
    if pd.isna(area_px):
        return None
    if area_px < small_thresh:
        return "small"
    elif area_px < medium_thresh:
        return "medium"
    else:
        return "large"


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", default=str(DEFAULT_DATA_DIR),
                         help="Folder root dataset (berisi data.yaml, train/, valid/, test/)")
    parser.add_argument("--out_dir", default=str(DEFAULT_OUT_DIR))
    args = parser.parse_args()

    data_dir = Path(args.data_dir).resolve()
    out_dir = Path(args.out_dir).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    data_yaml = data_dir / "data.yaml"
    class_names = load_class_names(data_yaml)
    print(f"Kelas terdeteksi: {class_names}")

    splits = ["train", "valid", "test"]
    all_dfs = {}

    for split in splits:
        split_dir = data_dir / split
        if not os.path.exists(split_dir):
            print(f"Peringatan: folder {split_dir} tidak ditemukan, dilewati.")
            continue
        df = parse_split(split_dir, class_names)
        df["split"] = split
        all_dfs[split] = df
        print(f"{split}: {df['image'].nunique()} gambar, "
              f"{df['class_id'].notna().sum()} bounding box")

    full_df = pd.concat(all_dfs.values(), ignore_index=True)

    # ---------------------------------------------------------------
    # Rasio split (persentase gambar per split) - supaya langsung
    # kelihatan dataset ini 70/20/10, 80/10/10, atau rasio lain
    # ---------------------------------------------------------------
    split_counts = {split: df["image"].nunique() for split, df in all_dfs.items()}
    total_images = sum(split_counts.values())

    print("\n" + "=" * 50)
    print("RASIO SPLIT DATASET")
    print("=" * 50)
    for split, count in split_counts.items():
        pct = (count / total_images * 100) if total_images else 0
        print(f"{split:6s}: {count:5d} gambar ({pct:.1f}%)")
    print(f"{'total':6s}: {total_images:5d} gambar")
    print("=" * 50)

    # ---------------------------------------------------------------
    # Tabel 2: jumlah gambar & box per kelas per split
    # ---------------------------------------------------------------
    box_counts = (
        full_df.dropna(subset=["class_name"])
        .groupby(["split", "class_name"])
        .size()
        .unstack(fill_value=0)
    )
    box_counts = box_counts.reindex(columns=class_names, fill_value=0)
    box_counts.to_csv(out_dir / "class_distribution.csv")
    print("\nTabel 2 - Jumlah bounding box per kelas per split:")
    print(box_counts)

    # ---------------------------------------------------------------
    # Gambar 3: Bar chart distribusi kelas (train vs valid vs test)
    # ---------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    box_counts.T.plot(kind="bar", ax=ax)
    ax.set_title("Distribusi Jumlah Bounding Box per Kelas per Split")
    ax.set_xlabel("Kelas Defect")
    ax.set_ylabel("Jumlah Bounding Box")
    ax.legend(title="Split")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(out_dir / "class_distribution.png", dpi=150)
    plt.close()

    # ---------------------------------------------------------------
    # Statistik ukuran bounding box (small/medium/large)
    # ---------------------------------------------------------------
    box_df = full_df.dropna(subset=["box_w_px", "box_h_px"]).copy()
    box_df["area_px"] = box_df["box_w_px"] * box_df["box_h_px"]
    box_df["size_category"] = box_df["area_px"].apply(categorize_box_size)
    box_df["aspect_ratio"] = box_df["box_w_px"] / box_df["box_h_px"]

    size_dist = box_df.groupby(["split", "size_category"]).size().unstack(fill_value=0)
    size_dist.to_csv(out_dir / "box_size_distribution.csv")
    print("\nDistribusi ukuran bounding box (small/medium/large):")
    print(size_dist)

    fig, ax = plt.subplots(figsize=(8, 5))
    size_dist.plot(kind="bar", stacked=True, ax=ax)
    ax.set_title("Distribusi Ukuran Bounding Box per Split")
    ax.set_xlabel("Split")
    ax.set_ylabel("Jumlah Box")
    plt.tight_layout()
    plt.savefig(out_dir / "box_size_distribution.png", dpi=150)
    plt.close()

    # ---------------------------------------------------------------
    # Histogram jumlah box per gambar
    # ---------------------------------------------------------------
    boxes_per_image = full_df.dropna(subset=["class_name"]).groupby(
        ["split", "image"]
    ).size().reset_index(name="box_count")

    fig, ax = plt.subplots(figsize=(8, 5))
    for split in boxes_per_image["split"].unique():
        subset = boxes_per_image[boxes_per_image["split"] == split]
        ax.hist(subset["box_count"], bins=range(1, subset["box_count"].max() + 2),
                alpha=0.5, label=split)
    ax.set_title("Distribusi Jumlah Bounding Box per Gambar")
    ax.set_xlabel("Jumlah Box dalam Satu Gambar")
    ax.set_ylabel("Frekuensi")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_dir / "boxes_per_image.png", dpi=150)
    plt.close()

    # ---------------------------------------------------------------
    # Ringkasan temuan (balanced/tidak) -> untuk teks Bab 2.3 / 3.1
    # ---------------------------------------------------------------
    train_counts = box_counts.loc["train"] if "train" in box_counts.index else None
    summary_lines = ["RINGKASAN EDA", "=" * 50]

    summary_lines.append("Rasio split dataset:")
    for split, count in split_counts.items():
        pct = (count / total_images * 100) if total_images else 0
        summary_lines.append(f"  {split}: {count} gambar ({pct:.1f}%)")
    summary_lines.append(f"  total: {total_images} gambar")
    summary_lines.append("")

    if train_counts is not None:
        max_class = train_counts.idxmax()
        min_class = train_counts.idxmin()
        imbalance_ratio = train_counts.max() / max(train_counts.min(), 1)

        summary_lines.append(
            f"Kelas dengan box terbanyak di train: {max_class} ({train_counts.max()} box)"
        )
        summary_lines.append(
            f"Kelas dengan box tersedikit di train: {min_class} ({train_counts.min()} box)"
        )
        summary_lines.append(f"Rasio imbalance (max/min): {imbalance_ratio:.2f}x")
        summary_lines.append(
            "Kesimpulan: " + (
                "Dataset TIDAK seimbang secara jumlah box per kelas, "
                "meskipun jumlah GAMBAR per kelas relatif seimbang."
                if imbalance_ratio > 1.3 else
                "Dataset relatif seimbang baik dari sisi gambar maupun box."
            )
        )

    summary_text = "\n".join(summary_lines)
    with open(out_dir / "summary.txt", "w", encoding="utf-8") as f:
        f.write(summary_text)

    print("\n" + summary_text)
    print(f"\nSemua output EDA tersimpan di: {out_dir}")

main()


In [ ]:
from IPython.display import Image, display

for image_name in (
    "class_distribution.png",
    "box_size_distribution.png",
    "boxes_per_image.png",
):
    display(Image(filename=str(EDA_DIR / image_name)))


In [ ]:
import sys
sys.argv = [
    "balancing",
    "--data_dir", str(DATA_DIR),
    "--out_dir", str(BALANCED_TRAIN_DIR),
    "--overwrite",
]

import os
import argparse
import shutil
import random
import yaml
import pandas as pd
import cv2
import albumentations as A
from collections import defaultdict
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
DEFAULT_DATA_DIR = PROJECT_DIR / "outputs" / "5_split" / "dataset_801010"
DEFAULT_OUT_DIR = PROJECT_DIR / "outputs" / "2_balancing_notebook" / "train_balanced"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def load_class_names(data_yaml_path):
    with open(data_yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    class_names = data.get("names")
    if not isinstance(class_names, list) or not class_names:
        raise ValueError(f"Daftar nama kelas tidak valid di: {data_yaml_path}")
    return class_names


def get_image_classes(labels_dir):
    """
    Mengembalikan dict: {nama_gambar: set(class_id yang muncul di gambar itu)}
    """
    image_classes = {}
    for label_file in os.listdir(labels_dir):
        if not label_file.endswith(".txt"):
            continue
        classes_in_image = set()
        with open(os.path.join(labels_dir, label_file), "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    classes_in_image.add(int(parts[0]))
        img_name = os.path.splitext(label_file)[0]
        image_classes[img_name] = classes_in_image
    return image_classes


def count_boxes_per_class(labels_dir, num_classes):
    counts = defaultdict(int)
    for label_file in os.listdir(labels_dir):
        if not label_file.endswith(".txt"):
            continue
        with open(os.path.join(labels_dir, label_file), "r", encoding="utf-8") as f:
            for line_number, line in enumerate(f, start=1):
                parts = line.strip().split()
                if not parts:
                    continue
                if len(parts) != 5:
                    raise ValueError(
                        f"Label harus berisi 5 kolom: "
                        f"{label_file}:{line_number}"
                    )
                try:
                    class_id = int(parts[0])
                    coordinates = [float(value) for value in parts[1:]]
                except ValueError as exc:
                    raise ValueError(
                        f"Format label tidak valid: {label_file}:{line_number}"
                    ) from exc
                if not 0 <= class_id < num_classes:
                    raise ValueError(
                        f"ID kelas di luar rentang: {label_file}:{line_number}"
                    )
                if not all(0 <= value <= 1 for value in coordinates):
                    raise ValueError(
                        f"Koordinat YOLO harus berada pada rentang 0-1: "
                        f"{label_file}:{line_number}"
                    )
                counts[class_id] += 1
    return counts


def get_augmentation_pipeline():
    """
    Augmentasi RINGAN untuk oversampling, supaya duplikat tidak identik
    pixel-per-pixel (membantu generalisasi, bukan cuma duplikasi murni).
    bbox_params memastikan koordinat box ikut ditransformasi dengan benar.
    """
    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomBrightnessContrast(p=0.4),
            A.Rotate(limit=10, p=0.4, border_mode=cv2.BORDER_CONSTANT),
        ],
        bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"]),
    )


def read_yolo_label(label_path):
    boxes, labels = [], []
    with open(label_path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) != 5:
                raise ValueError(
                    f"Label harus berisi 5 kolom: {label_path}:{line_number}"
                )
            try:
                labels.append(int(parts[0]))
                boxes.append([float(x) for x in parts[1:5]])
            except ValueError as exc:
                raise ValueError(
                    f"Format label tidak valid: {label_path}:{line_number}"
                ) from exc
    return boxes, labels


def write_yolo_label(label_path, boxes, labels):
    with open(label_path, "w") as f:
        for box, label in zip(boxes, labels):
            f.write(f"{label} {' '.join(f'{v:.6f}' for v in box)}\n")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", type=Path, default=DEFAULT_DATA_DIR,
                        help="Folder dataset hasil split 80/10/10")
    parser.add_argument("--out_dir", type=Path, default=DEFAULT_OUT_DIR,
                        help="Folder output train yang sudah diseimbangkan")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--overwrite", action="store_true",
                        help="Ganti isi folder output balancing yang sudah ada")
    args = parser.parse_args()

    random.seed(args.seed)

    data_dir = args.data_dir.resolve()
    data_yaml = data_dir / "data.yaml"
    if not data_yaml.is_file():
        raise FileNotFoundError(f"Konfigurasi dataset tidak ditemukan: {data_yaml}")
    class_names = load_class_names(data_yaml)
    num_classes = len(class_names)

    train_images_dir = data_dir / "train" / "images"
    train_labels_dir = data_dir / "train" / "labels"
    if not train_images_dir.is_dir():
        raise FileNotFoundError(f"Folder gambar train tidak ditemukan: {train_images_dir}")
    if not train_labels_dir.is_dir():
        raise FileNotFoundError(f"Folder label train tidak ditemukan: {train_labels_dir}")

    out_dir = args.out_dir.resolve()
    if (out_dir == data_dir or out_dir in data_dir.parents
            or data_dir in out_dir.parents):
        raise ValueError("Folder output harus terpisah dari folder dataset sumber.")
    if out_dir.exists() and not out_dir.is_dir():
        raise NotADirectoryError(f"Folder output bukan direktori: {out_dir}")
    overwrite = args.overwrite or out_dir == DEFAULT_OUT_DIR.resolve()
    if out_dir.exists() and any(out_dir.iterdir()):
        if not overwrite:
            raise FileExistsError(
                f"Folder output sudah berisi file: {out_dir}. "
                "Pilih --out_dir lain atau tambahkan --overwrite."
            )
        shutil.rmtree(out_dir)

    out_images_dir = out_dir / "images"
    out_labels_dir = out_dir / "labels"
    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)
    image_paths = sorted(
        path for path in train_images_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        raise ValueError(f"Tidak ada gambar train di: {train_images_dir}")
    image_paths_by_stem = {path.stem: path for path in image_paths}
    if len(image_paths_by_stem) != len(image_paths):
        raise ValueError("Ada beberapa gambar train dengan nama dasar yang sama.")
    missing_labels = [
        path for path in image_paths
        if not (train_labels_dir / f"{path.stem}.txt").is_file()
    ]
    if missing_labels:
        raise FileNotFoundError(
            f"Label tidak ditemukan untuk {len(missing_labels)} gambar; "
            f"contoh: {missing_labels[0].name}"
        )

    # --- Step 1: salin dulu seluruh train asli ke folder balanced ---
    print("Menyalin train set asli ke folder balanced...")
    for img_path in image_paths:
        shutil.copy2(img_path, out_images_dir / img_path.name)
        shutil.copy2(
            train_labels_dir / f"{img_path.stem}.txt",
            out_labels_dir / f"{img_path.stem}.txt",
        )

    # --- Step 2: hitung distribusi box SEBELUM balancing ---
    before_counts = count_boxes_per_class(train_labels_dir, num_classes)
    print("\nJumlah box per kelas SEBELUM balancing:")
    for cid, cname in enumerate(class_names):
        print(f"  {cname}: {before_counts.get(cid, 0)}")

    target_count = max(before_counts.values(), default=0)
    if target_count == 0:
        raise ValueError("Train set tidak memiliki bounding box yang dapat diseimbangkan.")
    print(f"\nTarget jumlah box per kelas (mengikuti kelas mayoritas): {target_count}")

    # --- Step 3: cari gambar per kelas untuk oversampling ---
    image_classes = get_image_classes(train_labels_dir)
    images_by_class = defaultdict(list)
    for img_name, classes in image_classes.items():
        for cid in classes:
            images_by_class[cid].append(img_name)

    transform = get_augmentation_pipeline()
    after_counts = dict(before_counts)

    # --- Step 4: oversampling untuk kelas yang kurang dari target ---
    for cid, cname in enumerate(class_names):
        current = before_counts.get(cid, 0)
        if current >= target_count or current == 0:
            continue

        deficit = target_count - current
        candidate_images = images_by_class.get(cid, [])
        if not candidate_images:
            print(f"  Peringatan: tidak ada gambar untuk kelas {cname}, dilewati.")
            continue

        # perkiraan rata-rata box kelas ini per gambar kandidat
        avg_box_per_image = max(current / len(candidate_images), 1e-6)
        n_duplicates_needed = max(1, int(deficit / avg_box_per_image))

        print(f"Oversampling kelas '{cname}': butuh ~{n_duplicates_needed} duplikat gambar")

        for i in range(n_duplicates_needed):
            src_img_name = random.choice(candidate_images)

            src_img_path = image_paths_by_stem.get(src_img_name)
            if src_img_path is None:
                continue
            label_path = train_labels_dir / f"{src_img_name}.txt"

            image = cv2.imread(str(src_img_path))
            if image is None:
                raise ValueError(f"Gambar tidak dapat dibaca: {src_img_path}")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            boxes, labels = read_yolo_label(label_path)

            try:
                augmented = transform(image=image, bboxes=boxes, class_labels=labels)
            except (ValueError, RuntimeError) as exc:
                print(f"  Lewati {src_img_name} karena error augmentasi: {exc}")
                continue

            new_name = f"{src_img_name}_class{cid}_aug{i}"
            new_img_path = out_images_dir / f"{new_name}{src_img_path.suffix}"
            new_label_path = out_labels_dir / f"{new_name}.txt"

            aug_image_bgr = cv2.cvtColor(augmented["image"], cv2.COLOR_RGB2BGR)
            if not cv2.imwrite(str(new_img_path), aug_image_bgr):
                raise OSError(f"Gambar hasil augmentasi gagal disimpan: {new_img_path}")
            write_yolo_label(new_label_path, augmented["bboxes"], augmented["class_labels"])

            for l in augmented["class_labels"]:
                after_counts[l] = after_counts.get(l, 0) + 1

    # --- Step 5: laporan before vs after (Tabel 3) ---
    report_rows = []
    for cid, cname in enumerate(class_names):
        report_rows.append({
            "class_name": cname,
            "before": before_counts.get(cid, 0),
            "after": after_counts.get(cid, 0),
        })
    report_df = pd.DataFrame(report_rows)
    report_path = out_dir.parent / "balancing_report.csv"
    report_df.to_csv(report_path, index=False)

    print("\nTabel 3 - Before vs After Balancing:")
    print(report_df)
    print(f"\nDataset balanced tersimpan di: {out_dir}")
    print(f"Laporan balancing tersimpan di: {report_path}")

    # --- Step 6: buat data.yaml khusus untuk versi balanced ---
    with open(data_yaml, "r", encoding="utf-8") as f:
        data_cfg = yaml.safe_load(f)

    balanced_yaml = dict(data_cfg)
    balanced_yaml["train"] = str(out_images_dir.resolve())
    balanced_yaml["path"] = str(data_dir)
    balanced_yaml_path = out_dir.parent / "data_balanced.yaml"
    with open(balanced_yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(balanced_yaml, f, sort_keys=False)
    print(f"data.yaml untuk training balanced tersimpan di: {balanced_yaml_path}")

main()


In [ ]:
import pandas as pd
from IPython.display import display

display(pd.read_csv(BALANCING_DIR / "balancing_report.csv"))


In [ ]:
import sys
sys.argv = [
    "train",
    "--data_dir", str(DATA_DIR),
    "--balanced_data", str(BALANCING_DIR / "data_balanced.yaml"),
    "--out_dir", str(TRAIN_DIR),
    "--model", MODEL,
    "--epochs", str(EPOCHS),
    "--imgsz", str(IMGSZ),
    "--batch", str(BATCH),
    "--patience", str(PATIENCE),
]

import argparse
from pathlib import Path

import pandas as pd
from ultralytics import YOLO

PROJECT_DIR = Path.cwd().resolve()
DEFAULT_DATA_DIR = PROJECT_DIR / "outputs" / "5_split" / "dataset_801010"
DEFAULT_OUTPUT_DIR = PROJECT_DIR / "outputs" / "3_train"
DEFAULT_BALANCED_YAML = PROJECT_DIR / "outputs" / "2_balancing_notebook" / "data_balanced.yaml"


def train_scenario(data_yaml_path, model_name, run_name, output_dir,
                   epochs, imgsz, batch, patience):
    """
    Melatih satu skenario (unbalanced atau balanced) dengan konfigurasi
    yang identik, kecuali data.yaml yang dipakai.
    """
    print(f"\n{'=' * 60}")
    print(f"MEMULAI TRAINING: {run_name}")
    print(f"Dataset config : {data_yaml_path}")
    print(f"{'=' * 60}\n")

    model = YOLO(model_name)  # pretrained COCO weights (transfer learning)

    train_args = {
        "data": str(data_yaml_path),
        "epochs": epochs,
        "imgsz": imgsz,
        "batch": batch,
        "patience": patience,
        "optimizer": "auto",
        "lr0": 0.01,
        "lrf": 0.01,
        "momentum": 0.937,
        "weight_decay": 0.0005,
        "warmup_epochs": 3,
        # Augmentasi bawaan Ultralytics (dicatat juga di Bab 2.6 paper):
        "hsv_h": 0.015,
        "hsv_s": 0.7,
        "hsv_v": 0.4,
        "fliplr": 0.5,
        "flipud": 0.0,
        "translate": 0.1,
        "scale": 0.5,
        "mosaic": 1.0,
        "close_mosaic": 10,
        "project": str(output_dir),
        "name": run_name,
        "exist_ok": True,
        "seed": 42,
        "verbose": True,
    }

    results = model.train(**train_args)
    return model, results


def evaluate_on_split(model, data_yaml_path, split, run_name, output_dir):
    """
    Menjalankan model.val() pada split tertentu (valid/test) dan
    mencetak metrik per kelas - untuk Tabel 5 & 6 di paper.
    """
    print(f"\nEvaluasi pada split '{split}' untuk {run_name}...")
    # Ultralytics membaca kunci 'val' di data.yaml, bukan 'valid'
    ul_split = "val" if split == "valid" else split
    metrics = model.val(
        data=str(data_yaml_path),
        split=ul_split,
        save_json=True,
        project=str(output_dir / run_name),
        name=f"{split}_evaluation",
        exist_ok=True,
    )

    print(f"mAP50     : {metrics.box.map50:.3f}")
    print(f"mAP50-95  : {metrics.box.map:.3f}")
    print(f"Precision : {metrics.box.mp:.3f}")
    print(f"Recall    : {metrics.box.mr:.3f}")

    return metrics


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", type=Path, default=DEFAULT_DATA_DIR)
    parser.add_argument("--out_dir", type=Path, default=DEFAULT_OUTPUT_DIR,
                        help="Folder utama untuk semua output training")
    parser.add_argument("--balanced_data", type=Path, default=DEFAULT_BALANCED_YAML,
                        help="Path YAML dataset balanced dari notebook")
    parser.add_argument("--model", default="yolov8n.pt",
                         help="yolov8n.pt / yolov8s.pt / yolov8m.pt untuk benchmark ukuran model")
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--batch", type=int, default=16)
    parser.add_argument("--patience", type=int, default=20)
    args = parser.parse_args()

    data_dir = args.data_dir.resolve()
    output_dir = args.out_dir.resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    unbalanced_yaml = data_dir / "data.yaml"
    balanced_yaml = args.balanced_data.resolve()
    if not unbalanced_yaml.is_file():
        raise FileNotFoundError(f"Konfigurasi dataset tidak ditemukan: {unbalanced_yaml}")
    model_tag = Path(args.model).stem

    scenarios = [
        ("unbalanced", unbalanced_yaml),
        ("balanced", balanced_yaml),
    ]

    summary = []

    for label, yaml_path in scenarios:
        if not yaml_path.is_file():
            print(f"Lewati skenario '{label}': {yaml_path} tidak ditemukan.")
            continue

        run_name = f"{label}_{model_tag}"
        model, _ = train_scenario(
            yaml_path, args.model, run_name, output_dir,
            args.epochs, args.imgsz, args.batch, args.patience
        )

        # Evaluasi pada valid dan test set pakai best.pt yang baru dilatih
        # Pakai folder simpan yang sebenarnya dari trainer (lebih aman daripada
        # path yang ditulis manual, karena lokasi runs/ bisa berbeda antar versi)
        save_dir = Path(model.trainer.save_dir)
        best_weights = save_dir / "weights" / "best.pt"
        if not best_weights.is_file():
            raise FileNotFoundError(f"Bobot terbaik tidak ditemukan: {best_weights}")
        eval_model = YOLO(str(best_weights))

        val_metrics = evaluate_on_split(
            eval_model, unbalanced_yaml, "valid", run_name, output_dir
        )
        test_metrics = evaluate_on_split(
            eval_model, unbalanced_yaml, "test", run_name, output_dir
        )
        # Catatan: valid/test SELALU pakai unbalanced_yaml karena kedua skenario
        # berbagi valid/test set yang sama (tidak ikut di-balancing)

        summary.append({
            "scenario": label,
            "model": args.model,
            "run_dir": str(save_dir),
            "val_mAP50": val_metrics.box.map50,
            "val_mAP50-95": val_metrics.box.map,
            "test_mAP50": test_metrics.box.map50,
            "test_mAP50-95": test_metrics.box.map,
            "test_precision": test_metrics.box.mp,
            "test_recall": test_metrics.box.mr,
        })

    # --- Tabel 7: Perbandingan Eksperimen (balanced vs unbalanced) ---
    if not summary:
        raise FileNotFoundError(
            f"Tidak ada konfigurasi dataset yang dapat dilatih. "
            f"Periksa {unbalanced_yaml} dan {balanced_yaml}."
        )
    summary_df = pd.DataFrame(summary)
    summary_path = output_dir / "experiment_comparison.csv"
    summary_df.to_csv(summary_path, index=False)
    print("\n" + "=" * 60)
    print("RINGKASAN PERBANDINGAN EKSPERIMEN (Tabel 7)")
    print("=" * 60)
    print(summary_df.to_string(index=False))
    print(f"\nTersimpan di: {summary_path}")

main()


In [ ]:
import sys
if not WEIGHTS_PATH.is_file():
    raise FileNotFoundError(f"Bobot model tidak ditemukan: {WEIGHTS_PATH}")
sys.argv = [
    "evaluate",
    "--weights", str(WEIGHTS_PATH),
    "--data_dir", str(DATA_DIR),
    "--split", "test",
    "--out_dir", str(EVALUATE_DIR),
]

import argparse
import math
import random
from pathlib import Path

import yaml
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO

PROJECT_DIR = Path.cwd().resolve()
DEFAULT_DATA_DIR = PROJECT_DIR / "outputs" / "5_split" / "dataset_801010"
DEFAULT_OUT_DIR = PROJECT_DIR / "outputs" / "4_evaluate"


def per_class_metrics_table(metrics, class_names, out_csv):
    """
    Mengubah hasil metrics.box dari Ultralytics menjadi DataFrame
    per kelas (format Tabel 5/6 di paper: P, R, mAP50, mAP50-95).
    """
    rows = []
    # metrics.box.ap_class_index -> index kelas yang punya hasil evaluasi
    # metrics.box.p, .r, .ap50, .ap -> array per kelas
    for i, cls_idx in enumerate(metrics.box.ap_class_index):
        rows.append({
            "class_name": class_names[int(cls_idx)],
            "precision": metrics.box.p[i],
            "recall": metrics.box.r[i],
            "mAP50": metrics.box.ap50[i],
            "mAP50-95": metrics.box.ap[i],
        })

    df = pd.DataFrame(rows)

    # baris rata-rata keseluruhan (mengikuti format Tabel 1 Paskasius)
    avg_row = {
        "class_name": "Rata-rata (all classes)",
        "precision": metrics.box.mp,
        "recall": metrics.box.mr,
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
    }
    df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
    df = df.round(3)
    df.to_csv(out_csv, index=False)
    print(f"\nTabel per-kelas tersimpan di: {out_csv}")
    print(df.to_string(index=False))
    return df


def visualize_predictions(model, images_dir, class_names, out_path,
                           n_images=9, conf_threshold=0.25):
    """
    Membuat grid 3x3 visualisasi prediksi pada citra test secara acak,
    untuk Gambar 7 (grid prediksi, termasuk contoh gagal jika ada).
    """
    images_dir = Path(images_dir)
    if not images_dir.is_dir():
        raise FileNotFoundError(f"Folder gambar tidak ditemukan: {images_dir}")
    if n_images < 1:
        raise ValueError("n_images harus lebih besar dari 0")
    image_files = sorted(
        path for path in images_dir.iterdir()
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    if not image_files:
        raise ValueError(f"Tidak ada gambar yang dapat divisualisasikan di: {images_dir}")
    image_files = random.Random(42).sample(
        image_files, min(n_images, len(image_files))
    )

    grid_size = math.ceil(math.sqrt(len(image_files)))
    fig, axes = plt.subplots(
        grid_size, grid_size, figsize=(15, 15), squeeze=False
    )
    axes = axes.flatten()

    for idx, img_path in enumerate(image_files):
        results = model.predict(str(img_path), conf=conf_threshold, verbose=False)
        result = results[0]

        img = result.orig_img[:, :, ::-1]  # BGR -> RGB
        ax = axes[idx]
        ax.imshow(img)

        n_detections = len(result.boxes)
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])

            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor="lime", facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(x1, max(y1 - 5, 0),
                     f"{class_names[cls_id]} {conf:.2f}",
                     color="white", fontsize=8,
                     bbox=dict(facecolor="green", alpha=0.6, pad=1))

        title = img_path.name if n_detections > 0 else f"{img_path.name} (TIDAK TERDETEKSI)"
        ax.set_title(title, fontsize=9,
                     color="black" if n_detections > 0 else "red")
        ax.axis("off")

    # matikan axis kosong jika gambar kurang dari grid
    for idx in range(len(image_files), len(axes)):
        axes[idx].axis("off")

    plt.tight_layout()
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"Grid visualisasi prediksi tersimpan di: {out_path}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--weights", required=True, help="Path ke best.pt")
    parser.add_argument("--data_dir", type=Path, default=DEFAULT_DATA_DIR)
    parser.add_argument("--split", default="test", choices=["valid", "test"])
    parser.add_argument("--out_dir", type=Path, default=DEFAULT_OUT_DIR)
    parser.add_argument("--n_vis", type=int, default=9)
    args = parser.parse_args()

    data_dir = args.data_dir.resolve()
    data_yaml = data_dir / "data.yaml"
    if not data_yaml.is_file():
        raise FileNotFoundError(f"Konfigurasi dataset tidak ditemukan: {data_yaml}")
    with data_yaml.open("r", encoding="utf-8") as f:
        class_names = yaml.safe_load(f).get("names")
    if isinstance(class_names, dict):
        class_names = [class_names[key] for key in sorted(class_names, key=int)]
    if not isinstance(class_names, list) or not class_names:
        raise ValueError(f"Daftar nama kelas tidak valid di: {data_yaml}")
    if args.n_vis < 1:
        parser.error("--n_vis harus lebih besar dari 0")

    weights_path = Path(args.weights).resolve()
    if not weights_path.is_file():
        raise FileNotFoundError(f"Bobot model tidak ditemukan: {weights_path}")
    run_name = f"{weights_path.parent.parent.name}_{args.split}"
    out_dir = args.out_dir.resolve() / run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    model = YOLO(str(weights_path))

    # --- Evaluasi metrik per kelas ---
    # Ultralytics membaca kunci 'val' di data.yaml, bukan 'valid'
    ul_split = "val" if args.split == "valid" else args.split
    metrics = model.val(
        data=str(data_yaml),
        split=ul_split,
        save_json=True,
        project=str(out_dir),
        name="ultralytics",
        exist_ok=True,
    )
    out_csv = out_dir / f"metrics_{args.split}.csv"
    per_class_metrics_table(metrics, class_names, out_csv)

    # --- Visualisasi prediksi ---
    images_dir = data_dir / args.split / "images"
    out_vis_path = out_dir / f"prediction_grid_{args.split}.png"
    visualize_predictions(model, images_dir, class_names, out_vis_path,
                           n_images=args.n_vis)

    print(f"\nSemua output evaluasi tersimpan di: {out_dir}")
    print(f"Output tambahan Ultralytics tersimpan di: {out_dir / 'ultralytics'}")

main()


In [ ]:
from IPython.display import Image, display

prediction_grid = EVALUATE_DIR / f"{WEIGHTS_PATH.parent.parent.name}_test" / "prediction_grid_test.png"
display(Image(filename=str(prediction_grid)))
